In [30]:
import random
from collections import defaultdict

import pandas as pd
from IPython.display import HTML
from rdkit import Chem, RDLogger
from rdkit.Chem import PandasTools
from rdkit.Chem.Scaffolds import MurckoScaffold
from chembl_webresource_client.new_client import new_client

RDLogger.DisableLog("rdApp.*")

TARGET_CHEMBL_ID = "CHEMBL3192"
TEST_SIZE = 0.2
RANDOM_STATE = 42
STD_THRESHOLD = 0.5
ASSAY_CHUNK_SIZE = 50

# Search for target protein
Retrieve HDAC8 (`CHEMBL3192`) **binding** assays (`assay_type='B'`) with `confidence_score=9`, then collect IC50 activities for those assays.

In [31]:
assay_client = new_client.assay

assays = pd.DataFrame(
    assay_client.filter(
        target_chembl_id=TARGET_CHEMBL_ID,
        confidence_score=9,
        assay_type="B",
    ).only(
        [
            "assay_chembl_id",
            "target_chembl_id",
            "assay_type",
            "confidence_score",
            "confidence_description",
        ]
    )
)

print("Number of matching assays:", len(assays))
assays.head()

Number of matching assays: 882


,assay_chembl_id,assay_type,confidence_description,confidence_score,description,target_chembl_id
0,CHEMBL695893,B,Direct single protein target assigned,9,Inhibition of Histone deacetylase 8 (HDAC8) of...,CHEMBL3192
1,CHEMBL827903,B,Direct single protein target assigned,9,Inhibition of human histone deacetylase 8 prep...,CHEMBL3192
2,CHEMBL907065,B,Direct single protein target assigned,9,Inhibition of HDAC8 (mean IC50),CHEMBL3192
3,CHEMBL912072,B,Direct single protein target assigned,9,Inhibition of human HDAC8,CHEMBL3192
4,CHEMBL890787,B,Direct single protein target assigned,9,Inhibition of HDAC8 in HeLa cells,CHEMBL3192


In [35]:
assay_ids = assays["assay_chembl_id"].tolist()
activity_client = new_client.activity
activity_chunks = []

for start in range(0, len(assay_ids), ASSAY_CHUNK_SIZE):
    chunk_ids = assay_ids[start:start + ASSAY_CHUNK_SIZE]
    records = activity_client.filter(
        target_chembl_id=TARGET_CHEMBL_ID,
        standard_type="IC50",
        assay_chembl_id__in=chunk_ids,
    )
    activity_chunks.append(pd.DataFrame(records))

activities = pd.concat(activity_chunks, ignore_index=True)
activities = activities.merge(
    assays[["assay_chembl_id", "confidence_score", "confidence_description"]],
    on="assay_chembl_id",
    how="left",
)

print("Number of IC50 activity records:", len(activities))
print("\nAssay type distribution:")
print(assays["assay_type"].value_counts(dropna=False))
print("\nConfidence score distribution:")
print(assays["confidence_score"].value_counts(dropna=False))
print("\nStandard type:")
print(activities["standard_type"].value_counts(dropna=False))
print("All activities map to filtered assays:", activities["assay_chembl_id"].isin(assay_ids).all())
print("Target ChEMBL IDs:", activities["target_chembl_id"].unique())
activities[["assay_chembl_id", "confidence_score", "confidence_description"]].head()

Number of IC50 activity records: 4520

Assay type distribution:
assay_type
B    882
Name: count, dtype: int64

Confidence score distribution:
confidence_score
9    882
Name: count, dtype: int64

Standard type:
standard_type
IC50    4520
Name: count, dtype: int64
All activities map to filtered assays: True
Target ChEMBL IDs: ['CHEMBL3192']


,assay_chembl_id,confidence_score,confidence_description
0,CHEMBL695893,9,Direct single protein target assigned
1,CHEMBL695893,9,Direct single protein target assigned
2,CHEMBL695893,9,Direct single protein target assigned
3,CHEMBL695893,9,Direct single protein target assigned
4,CHEMBL695893,9,Direct single protein target assigned


# Handling data

Keep records with a numeric pChEMBL value and a valid, non-mixture SMILES string. Then aggregate replicate measurements of the same compound.

In [36]:
def smiles_to_inchi(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    inchi = Chem.MolToInchi(mol)
    return inchi if inchi else None


activities_clean = activities.loc[
    activities["canonical_smiles"].notna()
].copy()
activities_clean["pchembl_value"] = pd.to_numeric(
    activities_clean["pchembl_value"], errors="coerce"
)
activities_clean = activities_clean.dropna(subset=["pchembl_value"])
activities_clean = activities_clean[
    ~activities_clean["canonical_smiles"].str.contains(".", regex=False)
].copy()
activities_clean["inchi"] = activities_clean["canonical_smiles"].map(smiles_to_inchi)
activities_clean = activities_clean.dropna(subset=["inchi"])

print("Records after cleaning:", len(activities_clean))
print("Unique compounds before aggregation:", activities_clean["canonical_smiles"].nunique())
activities_clean.head()

Records after cleaning: 3390
Unique compounds before aggregation: 2661


,action_type,activity_comment,activity_id,activity_properties,assay_chembl_id,assay_description,assay_type,assay_variant_accession,assay_variant_mutation,bao_endpoint,...,text_value,toid,type,units,uo_units,upper_value,value,confidence_score,confidence_description,inchi
2,None,None,1270391,[],CHEMBL695893,Inhibition of Histone deacetylase 8 (HDAC8) of...,B,None,None,BAO_0000190,...,None,None,IC50,uM,UO_0000065,None,0.8,9,Direct single protein target assigned,InChI=1S/C27H27N5O4/c33-18-32(36)15-3-1-2-8-23...
3,None,None,1271622,[],CHEMBL695893,Inhibition of Histone deacetylase 8 (HDAC8) of...,B,None,None,BAO_0000190,...,None,None,IC50,uM,UO_0000065,None,0.69,9,Direct single protein target assigned,InChI=1S/C27H27N5O4/c33-18-32(36)13-7-1-2-10-2...
4,None,None,1275487,[],CHEMBL695893,Inhibition of Histone deacetylase 8 (HDAC8) of...,B,None,None,BAO_0000190,...,None,None,IC50,uM,UO_0000065,None,6.8,9,Direct single protein target assigned,InChI=1S/C13H18N2O3/c16-11-15(18)10-6-2-5-9-13...
5,None,None,1275492,[],CHEMBL695893,Inhibition of Histone deacetylase 8 (HDAC8) of...,B,None,None,BAO_0000190,...,None,None,IC50,uM,UO_0000065,None,0.78,9,Direct single protein target assigned,InChI=1S/C19H24N2O3/c22-15-21(24)13-7-3-1-2-4-...
6,None,None,1276796,[],CHEMBL695893,Inhibition of Histone deacetylase 8 (HDAC8) of...,B,None,None,BAO_0000190,...,None,None,IC50,uM,UO_0000065,None,9.7,9,Direct single protein target assigned,InChI=1S/C15H22N2O3/c18-13-17(20)12-6-2-5-11-1...


In [37]:
compounds = (
    activities_clean.groupby("canonical_smiles", as_index=False)
    .agg(
        molecule_chembl_id=("molecule_chembl_id", "first"),
        pchembl_value_mean=("pchembl_value", "mean"),
        pchembl_value_std=("pchembl_value", "std"),
        inchi=("inchi", "first"),
    )
)
compounds["pchembl_value_std"] = compounds["pchembl_value_std"].fillna(0)
compounds = compounds[
    [
        "molecule_chembl_id",
        "canonical_smiles",
        "pchembl_value_mean",
        "pchembl_value_std",
        "inchi",
    ]
]

print("Unique compounds:", len(compounds))
compounds.head()

Unique compounds: 2661


,molecule_chembl_id,canonical_smiles,pchembl_value_mean,pchembl_value_std,inchi
0,CHEMBL4569890,Brc1cncs1,5.25,0.0,InChI=1S/C3H2BrNS/c4-3-1-5-2-6-3/h1-2H
1,CHEMBL5177240,C#CCN(C)CCCOc1cc(NC(=O)CCCCCCC(=O)NO)c(Cl)cc1Cl,5.75,0.0,InChI=1S/C21H29Cl2N3O4/c1-3-11-26(2)12-8-13-30...
2,CHEMBL5170598,C#CCN(C)CCCOc1cc(NC(=O)CCCCCCC(=O)NO)ccc1Cl,7.30,0.0,InChI=1S/C21H30ClN3O4/c1-3-13-25(2)14-8-15-29-...
3,CHEMBL5187984,C#CCN(C)CCCOc1cc(NC(=O)CCCCCCCC(=O)NO)ccc1Cl,5.95,0.0,InChI=1S/C22H32ClN3O4/c1-3-14-26(2)15-9-16-30-...
4,CHEMBL5200566,C#CCN(C)CCCOc1cc(NC(=O)CCCCCCCCC(=O)NO)ccc1Cl,5.61,0.0,InChI=1S/C23H34ClN3O4/c1-3-15-27(2)16-10-17-31...


# Save to csv

In [38]:
compounds.to_csv("HDAC8_exp_data_inchi.csv", sep=",", index=False)

# Processing for QSAR

In [39]:
dataset = compounds.loc[compounds["pchembl_value_std"] < STD_THRESHOLD].copy()
dataset = dataset.drop(columns=["pchembl_value_std"]).reset_index(drop=True)

print("Compounds with pChEMBL SD <", STD_THRESHOLD, ":", len(dataset))
dataset.describe()

Compounds with pChEMBL SD < 0.5 : 2616


,pchembl_value_mean
count,2616.000000
mean,6.049619
std,0.885126
min,4.010000
25%,5.460000
50%,6.000000
75%,6.620000
max,9.990000


# Creating training and test samples
Use a **Bemis–Murcko scaffold split** (test fraction = 0.2). Entire scaffold groups are assigned to either the training or the test set, so no Murcko scaffold is shared between splits.

In [40]:
def murcko_scaffold_smiles(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    scaffold = MurckoScaffold.GetScaffoldForMol(mol)
    if scaffold is None or scaffold.GetNumAtoms() == 0:
        return Chem.MolToSmiles(mol)
    return Chem.MolToSmiles(scaffold)


def scaffold_split(df, smiles_col="canonical_smiles", test_size=0.2, random_state=42):
    df = df.reset_index(drop=True).copy()
    scaffolds = df[smiles_col].map(murcko_scaffold_smiles)
    if scaffolds.isna().any():
        raise ValueError("Failed to generate a Murcko scaffold for some molecules.")
    df["scaffold"] = scaffolds

    groups = defaultdict(list)
    for idx, scaffold in enumerate(df["scaffold"]):
        groups[scaffold].append(idx)

    group_indices = list(groups.values())
    rng = random.Random(random_state)
    rng.shuffle(group_indices)

    n_total = len(df)
    n_train_cutoff = int(round(n_total * (1.0 - test_size)))
    train_idx, test_idx = [], []
    for group in group_indices:
        if len(train_idx) + len(group) <= n_train_cutoff:
            train_idx.extend(group)
        else:
            test_idx.extend(group)

    data_train = df.iloc[train_idx].reset_index(drop=True)
    data_test = df.iloc[test_idx].reset_index(drop=True)
    return data_train, data_test


data_train, data_test = scaffold_split(
    dataset,
    smiles_col="canonical_smiles",
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
)

In [41]:
train_scaffolds = set(data_train["scaffold"])
test_scaffolds = set(data_test["scaffold"])

print(f"Training compounds: {len(data_train)} ({len(data_train) / len(dataset):.1%})")
print(f"Test compounds: {len(data_test)} ({len(data_test) / len(dataset):.1%})")
print("Unique scaffolds in training set:", len(train_scaffolds))
print("Unique scaffolds in test set:", len(test_scaffolds))
print("Shared scaffolds:", len(train_scaffolds & test_scaffolds))
print("\nTraining pChEMBL mean ± SD: "
      f"{data_train['pchembl_value_mean'].mean():.3f} ± {data_train['pchembl_value_mean'].std():.3f}")
print("Test pChEMBL mean ± SD: "
      f"{data_test['pchembl_value_mean'].mean():.3f} ± {data_test['pchembl_value_mean'].std():.3f}")

assert train_scaffolds.isdisjoint(test_scaffolds), "Scaffold leakage between train and test."
data_train.head()

Training compounds: 2093 (80.0%)
Test compounds: 523 (20.0%)
Unique scaffolds in training set: 1106
Unique scaffolds in test set: 224
Shared scaffolds: 0

Training pChEMBL mean ± SD: 6.020 ± 0.869
Test pChEMBL mean ± SD: 6.168 ± 0.938


,molecule_chembl_id,canonical_smiles,pchembl_value_mean,inchi,scaffold
0,CHEMBL4299447,CC[C@@H]1OC(=O)[C@@H](C)[C@H](O[C@H]2C[C@@](C)...,5.77,"InChI=1S/C53H90N6O14/c1-14-41-53(10,66)46(62)3...",O=C1C[C@H](O[C@H]2CCCCO2)C[C@H](O[C@@H]2CCCCO2...
1,CHEMBL5899207,CC(C)c1ccc(-c2cc3ccc(C(=O)NO)cc3s2)cc1,5.96,InChI=1S/C18H17NO2S/c1-11(2)12-3-5-13(6-4-12)1...,c1ccc(-c2cc3ccccc3s2)cc1
2,CHEMBL4454484,O=C(CCCCCCCNC(=O)c1c(-c2ccc(O)cc2)sc2cc(O)ccc1...,6.40,InChI=1S/C23H26N2O5S/c26-16-9-7-15(8-10-16)22-...,c1ccc(-c2cc3ccccc3s2)cc1
3,CHEMBL4533192,O=C(CCCCCCNC(=O)c1c(-c2ccc(O)cc2)sc2cc(O)ccc12)NO,6.27,InChI=1S/C22H24N2O5S/c25-15-8-6-14(7-9-15)21-2...,c1ccc(-c2cc3ccccc3s2)cc1
4,CHEMBL4283105,O=C(NO)c1ccc(Cn2c3c(c4cc(F)ccc42)C[S+]([O-])CC...,5.62,InChI=1S/C19H17FN2O3S/c20-14-5-6-17-15(9-14)16...,c1ccc(Cn2c3c(c4ccccc42)C[SH+]CC3)cc1


# Creating sdf files
The training set is written to `HDAC8_train.sdf` and the test set to `HDAC8_test.sdf`.

## Test set

In [42]:
PandasTools.AddMoleculeColumnToFrame(data_test, "canonical_smiles", "Molecule")
HTML(data_test.head().to_html())

,molecule_chembl_id,canonical_smiles,pchembl_value_mean,inchi,scaffold,Molecule
0,CHEMBL5184762,O=C(/C=C/C1CCN(Cc2csc3ccccc23)CC1)NO,5.89,"InChI=1S/C17H20N2O2S/c20-17(18-21)6-5-13-7-9-19(10-8-13)11-14-12-22-16-4-2-1-3-15(14)16/h1-6,12-13,21H,7-11H2,(H,18,20)/b6-5+",c1ccc2c(CN3CCCCC3)csc2c1,<rdkit.Chem.rdchem.Mol object at 0x00000179B3FC3A00>
1,CHEMBL4218901,CC(C)[C@H](NS(=O)(=O)c1cccc(C(=O)NO)c1)C(=O)Nc1ccccc1,4.47,"InChI=1S/C18H21N3O5S/c1-12(2)16(18(23)19-14-8-4-3-5-9-14)21-27(25,26)15-10-6-7-13(11-15)17(22)20-24/h3-12,16,21,24H,1-2H3,(H,19,23)(H,20,22)/t16-/m0/s1",O=C(CNS(=O)(=O)c1ccccc1)Nc1ccccc1,<rdkit.Chem.rdchem.Mol object at 0x00000179B3FC09E0>
2,CHEMBL4464527,CN(C(=O)CN(C)S(=O)(=O)c1ccc(F)cc1)c1ccc(C(=O)NO)cc1,6.24,"InChI=1S/C17H18FN3O5S/c1-20(27(25,26)15-9-5-13(18)6-10-15)11-16(22)21(2)14-7-3-12(4-8-14)17(23)19-24/h3-10,24H,11H2,1-2H3,(H,19,23)",O=C(CNS(=O)(=O)c1ccccc1)Nc1ccccc1,<rdkit.Chem.rdchem.Mol object at 0x00000179B3FC0430>
3,CHEMBL4213975,C[C@H](NS(=O)(=O)c1cccc(C(=O)NO)c1)C(=O)Nc1ccccc1,4.62,"InChI=1S/C16H17N3O5S/c1-11(15(20)17-13-7-3-2-4-8-13)19-25(23,24)14-9-5-6-12(10-14)16(21)18-22/h2-11,19,22H,1H3,(H,17,20)(H,18,21)/t11-/m0/s1",O=C(CNS(=O)(=O)c1ccccc1)Nc1ccccc1,<rdkit.Chem.rdchem.Mol object at 0x00000179B3FC02E0>
4,CHEMBL4206647,O=C(CNS(=O)(=O)c1cccc(C(=O)NO)c1)Nc1ccccc1,5.46,"InChI=1S/C15H15N3O5S/c19-14(17-12-6-2-1-3-7-12)10-16-24(22,23)13-8-4-5-11(9-13)15(20)18-21/h1-9,16,21H,10H2,(H,17,19)(H,18,20)",O=C(CNS(=O)(=O)c1ccccc1)Nc1ccccc1,<rdkit.Chem.rdchem.Mol object at 0x00000179B3FC0890>


In [43]:
sdf_properties = [col for col in data_test.columns if col != "Molecule"]
PandasTools.WriteSDF(
    data_test,
    "HDAC8_test.sdf",
    molColName="Molecule",
    properties=sdf_properties,
)

## Training set

In [44]:
PandasTools.AddMoleculeColumnToFrame(data_train, "canonical_smiles", "Molecule")
HTML(data_train.head().to_html())

,molecule_chembl_id,canonical_smiles,pchembl_value_mean,inchi,scaffold,Molecule
0,CHEMBL4299447,CC[C@@H]1OC(=O)[C@@H](C)[C@H](O[C@H]2C[C@@](C)(OC)[C@@H](O)[C@H](C)O2)[C@@H](C)[C@H](O[C@H]2O[C@@H](C)C[C@@H](N(C)C)[C@@H]2O)[C@](C)(O)C[C@H](C)CN(Cc2ccc(-c3cn(CCCCCCC(=O)NO)nn3)cc2)[C@H](C)[C@H](O)[C@@]1(C)O,5.77,"InChI=1S/C53H90N6O14/c1-14-41-53(10,66)46(62)35(6)58(29-37-20-22-38(23-21-37)39-30-59(56-54-39)24-18-16-15-17-19-42(60)55-67)28-31(2)26-51(8,65)48(73-50-44(61)40(57(11)12)25-32(3)69-50)33(4)45(34(5)49(64)71-41)72-43-27-52(9,68-13)47(63)36(7)70-43/h20-23,30-36,40-41,43-48,50,61-63,65-67H,14-19,24-29H2,1-13H3,(H,55,60)/t31-,32-,33+,34-,35+,36-,40+,41-,43-,44-,45+,46-,47-,48-,50+,51+,52+,53-/m0/s1",O=C1C[C@H](O[C@H]2CCCCO2)C[C@H](O[C@@H]2CCCCO2)CCCCN(Cc2ccc(-c3c[nH]nn3)cc2)CCCCO1,<rdkit.Chem.rdchem.Mol object at 0x00000179B3FC1620>
1,CHEMBL5899207,CC(C)c1ccc(-c2cc3ccc(C(=O)NO)cc3s2)cc1,5.96,"InChI=1S/C18H17NO2S/c1-11(2)12-3-5-13(6-4-12)16-9-14-7-8-15(18(20)19-21)10-17(14)22-16/h3-11,21H,1-2H3,(H,19,20)",c1ccc(-c2cc3ccccc3s2)cc1,<rdkit.Chem.rdchem.Mol object at 0x00000179B4E220A0>
2,CHEMBL4454484,O=C(CCCCCCCNC(=O)c1c(-c2ccc(O)cc2)sc2cc(O)ccc12)NO,6.40,"InChI=1S/C23H26N2O5S/c26-16-9-7-15(8-10-16)22-21(18-12-11-17(27)14-19(18)31-22)23(29)24-13-5-3-1-2-4-6-20(28)25-30/h7-12,14,26-27,30H,1-6,13H2,(H,24,29)(H,25,28)",c1ccc(-c2cc3ccccc3s2)cc1,<rdkit.Chem.rdchem.Mol object at 0x00000179B4E21150>
3,CHEMBL4533192,O=C(CCCCCCNC(=O)c1c(-c2ccc(O)cc2)sc2cc(O)ccc12)NO,6.27,"InChI=1S/C22H24N2O5S/c25-15-8-6-14(7-9-15)21-20(17-11-10-16(26)13-18(17)30-21)22(28)23-12-4-2-1-3-5-19(27)24-29/h6-11,13,25-26,29H,1-5,12H2,(H,23,28)(H,24,27)",c1ccc(-c2cc3ccccc3s2)cc1,<rdkit.Chem.rdchem.Mol object at 0x00000179B4E222D0>
4,CHEMBL4283105,O=C(NO)c1ccc(Cn2c3c(c4cc(F)ccc42)C[S+]([O-])CC3)cc1,5.62,"InChI=1S/C19H17FN2O3S/c20-14-5-6-17-15(9-14)16-11-26(25)8-7-18(16)22(17)10-12-1-3-13(4-2-12)19(23)21-24/h1-6,9,24H,7-8,10-11H2,(H,21,23)",c1ccc(Cn2c3c(c4ccccc42)C[SH+]CC3)cc1,<rdkit.Chem.rdchem.Mol object at 0x00000179B4E215B0>


In [45]:
sdf_properties = [col for col in data_train.columns if col != "Molecule"]
PandasTools.WriteSDF(
    data_train,
    "HDAC8_train.sdf",
    molColName="Molecule",
    properties=sdf_properties,
)